# SEQ BirdDex Machine Learning Notebook

## 1. Title / Project Context

**Project:** BIRDIDEX - South-East Queensland offline bird recognition system  
**Goal:** build a traceable image-classification workflow that can support a field cyberdeck without needing network access at inference time.  
**Dataset source summary:** local class indexes, region/species priors, open-license image metadata, reviewed local images, and generated species profiles under the repo-owned `data/` layout.  
**ML objective:** train and evaluate a bird species classifier that returns top-k predictions, confidence scores, and abstention states instead of forcing weak IDs.  
**Device objective:** support two field modes: Pi camera instant capture and Sony A7RV import/review.  
**Safety note:** this notebook must not contain committed secrets, private seed values, local absolute paths, downloaded media, model weights, caches, or generated datasets.

This follows a CAB420-style structure: state the decision, show the code, inspect the evidence, and leave explicit TODOs where real training or curated biological facts are not yet available.


## 2. Reproducibility Setup

The first executable cell resolves the repo root, loads local environment variables, and sets deterministic seeds. The seed can come from `BIRDIDEX_MASTER_SEED`; if it is not configured, the notebook uses a documented demo seed so that audit cells still run without exposing a private project seed.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shlex
import subprocess
import sys
from collections import Counter, defaultdict
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path
from typing import Any


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until the BIRDIDEX package and pyproject are found."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "birdidex").exists():
            return candidate
    raise RuntimeError("Could not locate the BIRDIDEX repo root from the current notebook path.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from dotenv import load_dotenv

    # Local runtime config may define provider keys or the project seed; values are never printed.
    load_dotenv(REPO_ROOT / ".env", override=False)
    load_dotenv(REPO_ROOT / ".env.local", override=True)
    DOTENV_AVAILABLE = True
except Exception as exc:  # noqa: BLE001 - notebook should report optional setup gaps cleanly
    DOTENV_AVAILABLE = False
    print(f"[info] dotenv not available or not loaded: {type(exc).__name__}: {exc}")

try:
    import numpy as np

    NUMPY_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    NUMPY_AVAILABLE = False
    np = None  # type: ignore[assignment]
    print(f"[info] numpy unavailable: {type(exc).__name__}: {exc}")

try:
    import pandas as pd

    PANDAS_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    PANDAS_AVAILABLE = False
    pd = None  # type: ignore[assignment]
    print(f"[info] pandas unavailable: {type(exc).__name__}: {exc}")

try:
    import matplotlib.pyplot as plt

    MATPLOTLIB_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    MATPLOTLIB_AVAILABLE = False
    plt = None  # type: ignore[assignment]
    print(f"[info] matplotlib unavailable: {type(exc).__name__}: {exc}")

try:
    from IPython.display import display
except Exception:  # pragma: no cover - only used in notebook UI
    display = print  # type: ignore[assignment]

try:
    import torch
    import torchvision

    TORCH_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    TORCH_AVAILABLE = False
    torch = None  # type: ignore[assignment]
    torchvision = None  # type: ignore[assignment]
    print(f"[info] torch/torchvision unavailable: {type(exc).__name__}: {exc}")

from birdidex import paths as bx_paths
from birdidex import images as bx_images
from birdidex import profiles as bx_profiles
from birdidex import splits as bx_splits
from birdidex import taxonomy as bx_taxonomy
from birdidex.infer import confidence_gate
from birdidex.observations import ObservationRecord, observation_json_schema
from birdidex.seed import PURPOSE_SPLITS, derive_seed

DEMO_SEED = 42
seed_text = os.getenv("BIRDIDEX_MASTER_SEED")


def coerce_seed(value: str) -> int:
    try:
        return int(value.strip())
    except ValueError:
        return int(hashlib.sha256(value.strip().encode("utf-8")).hexdigest(), 16) % (2**31)


if seed_text:
    MASTER_SEED = coerce_seed(seed_text)
    MASTER_SEED_SOURCE = "configured environment variable"
else:
    MASTER_SEED = DEMO_SEED
    MASTER_SEED_SOURCE = "safe demo default"

random.seed(MASTER_SEED)
if NUMPY_AVAILABLE:
    np.random.seed(MASTER_SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(MASTER_SEED)


def table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> Any:
    if PANDAS_AVAILABLE:
        return pd.DataFrame(rows, columns=columns)
    return rows


def show_table(rows: list[dict[str, Any]], columns: list[str] | None = None) -> Any:
    result = table(rows, columns)
    display(result)
    return result


ENVIRONMENT_SUMMARY = [
    {"item": "repo_root", "value": str(REPO_ROOT.relative_to(REPO_ROOT)) or "."},
    {"item": "python", "value": sys.version.split()[0]},
    {"item": "seed_source", "value": MASTER_SEED_SOURCE},
    {"item": "dotenv_loaded", "value": DOTENV_AVAILABLE},
    {"item": "numpy", "value": NUMPY_AVAILABLE},
    {"item": "pandas", "value": PANDAS_AVAILABLE},
    {"item": "matplotlib", "value": MATPLOTLIB_AVAILABLE},
    {"item": "torch", "value": TORCH_AVAILABLE},
]
show_table(ENVIRONMENT_SUMMARY)


## 3. Data Layout Audit

This audit checks for the files and folders the package expects today. Missing files do not crash the notebook; they produce the exact local commands that create or refresh the scaffold.


In [ ]:
DATA_ROOT = bx_paths.data_dir()
BIRDDEX_ROOT = DATA_ROOT / "processed" / "birddex"
IMAGES_ROOT = bx_paths.images_dir()
SPLITS_ROOT = IMAGES_ROOT / "splits"

EXPECTED_FILES = {
    "class_index.json": bx_paths.default_class_index_path(),
    "species_catalog.csv": BIRDDEX_ROOT / "species_catalog.csv",
    "region_species_presence.csv": BIRDDEX_ROOT / "region_species_presence.csv",
    "species_region_summary.json": BIRDDEX_ROOT / "species_region_summary.json",
    "rarity_scaffold.json": BIRDDEX_ROOT / "rarity_scaffold.json",
    "image_records.jsonl": bx_images.image_records_path(IMAGES_ROOT),
    "species_profiles.json": bx_profiles.species_profiles_path(),
}
EXPECTED_FOLDERS = {
    "data/images/raw/": IMAGES_ROOT / "raw",
    "data/images/processed/": IMAGES_ROOT / "processed",
    "data/images/splits/train/": SPLITS_ROOT / "train",
    "data/images/splits/val/": SPLITS_ROOT / "val",
    "data/images/splits/test/": SPLITS_ROOT / "test",
}

availability_rows = []
for label, path in EXPECTED_FILES.items():
    availability_rows.append(
        {"kind": "file", "name": label, "exists": path.exists(), "path": str(path.relative_to(REPO_ROOT))}
    )
for label, path in EXPECTED_FOLDERS.items():
    availability_rows.append(
        {"kind": "folder", "name": label, "exists": path.exists(), "path": str(path.relative_to(REPO_ROOT))}
    )

availability = show_table(availability_rows, ["kind", "name", "exists", "path"])
missing = [row for row in availability_rows if not row["exists"]]

if missing:
    print("\nMissing expected resources. Generate them with the ordered local commands below:")
    for command in [
        "uv sync --all-groups",
        "uv run birdidex doctor",
        "uv run birdidex images scaffold",
        "uv run birdidex images pipeline --species-limit 5 --per-class 25 --target-accepted 10 --max-edge 1024 --format jpg --quality 85",
        "uv run birdidex images report",
        "uv run birdidex audit dataset",
        "uv run birdidex profiles build",
    ]:
        print("  " + command)
else:
    print("All expected notebook inputs are present.")


## 4. Class Index Loading

The class index is the only authority for model output IDs, labels, and deterministic folder names. This cell validates required fields, ID uniqueness, label uniqueness, stable folders, and ambiguous taxa that should not be used as normal classifier classes.


In [ ]:
class_index_path = EXPECTED_FILES["class_index.json"]
classes = []
classes_df = table([])

if not class_index_path.exists():
    print("class_index.json is missing. Run: uv run birdidex images scaffold")
else:
    try:
        classes = bx_taxonomy.load_class_index(class_index_path)
    except Exception as exc:  # noqa: BLE001 - report validation failures as notebook output
        print(f"[error] class index validation failed: {type(exc).__name__}: {exc}")
        classes = []

if classes:
    rows = []
    for taxon in classes:
        rows.append(
            {
                "class_id": taxon.class_id,
                "label": taxon.label,
                "folder_name": taxon.folder_name,
                "common_name": taxon.common_name,
                "scientific_name": taxon.scientific_name,
                "ambiguous_taxon": taxon.is_ambiguous,
                "known_regions": "; ".join(taxon.known_regions),
            }
        )
    classes_df = table(rows)
    show_table(rows[:12])

    clean = [taxon for taxon in classes if not taxon.is_ambiguous]
    ambiguous = [taxon for taxon in classes if taxon.is_ambiguous]
    folder_mismatches = [
        taxon.folder_name
        for taxon in classes
        if taxon.folder_name != bx_taxonomy.class_folder_name(taxon.class_id, taxon.label)
    ]
    summary = [
        {"check": "class rows", "value": len(classes)},
        {"check": "clean classifier classes", "value": len(clean)},
        {"check": "ambiguous taxa", "value": len(ambiguous)},
        {"check": "unique class IDs", "value": len({t.class_id for t in classes}) == len(classes)},
        {"check": "unique labels", "value": len({t.label for t in classes}) == len(classes)},
        {"check": "deterministic folder names", "value": not folder_mismatches},
    ]
    show_table(summary)
    if ambiguous:
        print("Ambiguous taxa are excluded from default image collection and classifier training:")
        show_table([
            {
                "class_id": t.class_id,
                "folder_name": t.folder_name,
                "common_name": t.common_name,
                "scientific_name": t.scientific_name,
            }
            for t in ambiguous[:20]
        ])
else:
    print("No class rows loaded, so downstream cells will run in audit-only mode.")


## 5. Region and Species Prior Audit

Regional and seasonal priors are context signals. The visual classifier proposes species from the image; the context prior can re-rank plausible classes, but it must not override strong visual evidence. This matters in field use because nearby similar species may share habitat while rare or out-of-season sightings still happen.


In [ ]:
region_presence_path = EXPECTED_FILES["region_species_presence.csv"]
species_summary_path = EXPECTED_FILES["species_region_summary.json"]

region_rows: list[dict[str, Any]] = []
if classes:
    for taxon in classes:
        region_rows.append(
            {
                "class_id": taxon.class_id,
                "label": taxon.label,
                "known_region_count": len(taxon.known_regions),
                "known_regions": "; ".join(taxon.known_regions[:4]),
                "ambiguous_taxon": taxon.is_ambiguous,
            }
        )

if region_presence_path.exists() and PANDAS_AVAILABLE:
    presence_df = pd.read_csv(region_presence_path)
    print(f"Loaded region presence table with {len(presence_df)} rows from {region_presence_path.relative_to(REPO_ROOT)}")
    display(presence_df.head(10))
elif region_presence_path.exists():
    print(f"Region presence table exists: {region_presence_path.relative_to(REPO_ROOT)}")
else:
    print("Region presence CSV missing. Rebuild the processed dataset before relying on priors.")

if species_summary_path.exists():
    summary_payload = json.loads(species_summary_path.read_text(encoding="utf-8"))
    print(f"Species-region summary loaded: {species_summary_path.relative_to(REPO_ROOT)}")
    if isinstance(summary_payload, dict):
        print(f"Top-level keys: {sorted(summary_payload)[:12]}")

if region_rows:
    show_table(sorted(region_rows, key=lambda row: row["known_region_count"], reverse=True)[:15])

print("Prior policy: visual classifier -> optional region/season re-rank -> confidence gate -> user-visible result.")


## 6. Image Metadata Audit

`image_records.jsonl` is the audit log for candidate, accepted, rejected, and quarantined image records. The notebook summarizes coverage, providers, licenses, resolutions, and duplicates when those fields are present.


In [ ]:
image_records_path = EXPECTED_FILES["image_records.jsonl"]
image_records = bx_images.read_metadata_jsonl(image_records_path)
record_dicts = [record.to_dict() for record in image_records]

if not record_dicts:
    print("No image metadata exists yet. Generate a small dataset run with:")
    for command in [
        "uv run birdidex providers doctor",
        "uv run birdidex providers dry-run --provider inaturalist --species \"Rainbow Lorikeet\" --limit 5",
        "uv run birdidex images pipeline --species-limit 5 --per-class 25 --target-accepted 10 --max-edge 1024 --format jpg --quality 85",
        "uv run birdidex images report",
    ]:
        print("  " + command)
else:
    print(f"Loaded {len(record_dicts)} image metadata records from {image_records_path.relative_to(REPO_ROOT)}")
    records_df = table(record_dicts)
    if PANDAS_AVAILABLE:
        display(records_df.head(5))

    accepted_counts = Counter(r["class_id"] for r in record_dicts if r.get("validation_status") == "accepted" or r.get("status") == "accepted")
    rejected_counts = Counter(r["class_id"] for r in record_dicts if r.get("validation_status") in {"quarantine", "rejected"} or r.get("status") in {"quarantine", "rejected"})
    provider_counts = Counter(r.get("provider") or "unknown" for r in record_dicts)
    license_counts = Counter(r.get("license_code") or "unknown" for r in record_dicts)
    resolution_counts = Counter()
    sha_counts = Counter(r.get("sha256") for r in record_dicts if r.get("sha256"))
    phash_counts = Counter(r.get("phash") for r in record_dicts if r.get("phash"))

    for row in record_dicts:
        width = row.get("stored_width") or row.get("original_width") or row.get("width")
        height = row.get("stored_height") or row.get("original_height") or row.get("height")
        if not width or not height:
            resolution_counts["unknown"] += 1
            continue
        longest = max(int(width), int(height))
        if longest <= 640:
            resolution_counts["<=640"] += 1
        elif longest <= 1280:
            resolution_counts["641-1280"] += 1
        elif longest <= 1920:
            resolution_counts["1281-1920"] += 1
        else:
            resolution_counts[">1920"] += 1

    show_table([
        {"metric": "records", "value": len(record_dicts)},
        {"metric": "accepted", "value": sum(accepted_counts.values())},
        {"metric": "rejected_or_quarantined", "value": sum(rejected_counts.values())},
        {"metric": "sha256 duplicate groups", "value": sum(1 for c in sha_counts.values() if c > 1)},
        {"metric": "phash duplicate groups", "value": sum(1 for c in phash_counts.values() if c > 1)},
    ])
    show_table([{"provider": key, "records": value} for key, value in provider_counts.most_common()])
    show_table([{"license": key, "records": value} for key, value in license_counts.most_common()])
    show_table([{"resolution_bucket": key, "records": value} for key, value in sorted(resolution_counts.items())])

    if MATPLOTLIB_AVAILABLE and provider_counts:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.bar(list(provider_counts), list(provider_counts.values()))
        ax.set_title("Provider distribution")
        ax.set_ylabel("records")
        ax.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()


## 7. Image Quality and Preprocessing Logic

Ingestion keeps useful detail up to 1024 px longest edge so beak, eye, and plumage features remain inspectable. Model transforms happen later because each architecture has its own input size and normalization. Resizing with aspect-ratio preservation plus letterboxing avoids stretching fine-grained bird features.


In [ ]:
from birdidex.images import (
    basic_sharpness_score,
    image_dimensions,
    inspect_image,
    letterbox_image,
    open_image_rgb,
    resize_with_aspect,
    validate_image_can_open,
)

sample_images = []
for root in [IMAGES_ROOT / "raw", IMAGES_ROOT / "processed"]:
    if root.exists():
        sample_images.extend(path for path in root.rglob("*") if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
sample_images = sorted(sample_images)[:5]

if sample_images:
    rows = []
    for path in sample_images:
        inspection = inspect_image(path)
        rows.append(asdict(inspection) | {"repo_path": str(path.relative_to(REPO_ROOT))})
    show_table(rows)

    first_image = open_image_rgb(sample_images[0])
    resized = resize_with_aspect(first_image, 256)
    boxed = letterbox_image(first_image, 224)
    print(f"Example preprocessing for {sample_images[0].relative_to(REPO_ROOT)}")
    print(f"  original: {first_image.size}; resized: {resized.size}; letterboxed: {boxed.size}")
    print(f"  sharpness score: {basic_sharpness_score(first_image):.2f}")
else:
    print("No real local images were found. Running a tiny in-memory preprocessing example instead.")
    try:
        from PIL import Image, ImageDraw

        demo = Image.new("RGB", (320, 180), (20, 20, 20))
        draw = ImageDraw.Draw(demo)
        draw.ellipse((110, 30, 210, 130), fill=(180, 120, 40))
        resized = resize_with_aspect(demo, 128)
        boxed = letterbox_image(demo, 224)
        print(f"  demo original: {demo.size}; resized: {resized.size}; letterboxed: {boxed.size}")
        print(f"  demo sharpness score: {basic_sharpness_score(demo):.2f}")
    except Exception as exc:  # noqa: BLE001
        print(f"Image helper demo skipped: {type(exc).__name__}: {exc}")


## 8. Split Audit and Creation Logic

Train/validation/test splits must be deterministic and class-stratified. Duplicate records should stay in the same split whenever `sha256`, perceptual hash, provider ID, or URL grouping data is available.

Coverage buckets used here:

- `<20`: smoke-test only
- `20-49`: weak
- `50-99`: minimal
- `100-199`: usable
- `200+`: preferred


In [ ]:
def coverage_bucket(count: int) -> str:
    if count < 20:
        return "<20 smoke-test only"
    if count < 50:
        return "20-49 weak"
    if count < 100:
        return "50-99 minimal"
    if count < 200:
        return "100-199 usable"
    return "200+ preferred"


def count_split_files(split_dir: Path) -> Counter[str]:
    counts: Counter[str] = Counter()
    if not split_dir.exists():
        return counts
    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            counts[class_dir.name] = sum(
                1 for path in class_dir.rglob("*") if path.is_file() or path.is_symlink()
            )
    return counts

split_counts = {split: count_split_files(SPLITS_ROOT / split) for split in bx_images.SPLIT_NAMES}
rows = []
folders = sorted({folder for counts in split_counts.values() for folder in counts})
for folder in folders:
    train_count = split_counts["train"].get(folder, 0)
    val_count = split_counts["val"].get(folder, 0)
    test_count = split_counts["test"].get(folder, 0)
    total = train_count + val_count + test_count
    rows.append(
        {
            "folder_name": folder,
            "train": train_count,
            "val": val_count,
            "test": test_count,
            "total": total,
            "coverage_bucket": coverage_bucket(total),
        }
    )

if rows:
    show_table(sorted(rows, key=lambda row: row["total"])[:20])
else:
    print("No split images detected. Create deterministic splits with:")
    print("  uv run birdidex images split --train 0.75 --val 0.15 --test 0.10 --seed-env BIRDIDEX_MASTER_SEED")

if image_records:
    accepted_with_hash = [record for record in image_records if record.status == "accepted" and (record.sha256 or record.phash)]
    print(f"Accepted records with duplicate keys available: {len(accepted_with_hash)}")
    if accepted_with_hash:
        split_seed = derive_seed(PURPOSE_SPLITS, master=MASTER_SEED)
        split_map = bx_splits.assign_split_names(accepted_with_hash, seed=split_seed)
        print(f"Deterministic duplicate grouping produced {len(split_map)} split groups.")
else:
    print("Duplicate leakage audit is waiting for image_records.jsonl.")


## 9. Baseline Model Scaffold

The device workflow eventually needs two model roles:

- **Detector/cropper:** find bird regions or reject multi-subject frames.
- **Classifier:** classify a cropped bird into top-k species candidates.

Top-k outputs matter because similar SEQ birds may require user confirmation. Confidence calibration matters because the field UI should abstain on uncertain or out-of-set images. Ambiguous taxa are excluded from normal classifier classes because they are not stable species-level labels.


In [ ]:
RUN_TRAINING_SMOKE_TEST = os.getenv("BIRDIDEX_RUN_TRAINING_SMOKE_TEST") == "1"
ALLOW_PRETRAINED_DOWNLOAD = os.getenv("BIRDIDEX_ALLOW_PRETRAINED_DOWNLOAD") == "1"

split_dirs_ready = all((SPLITS_ROOT / split).exists() for split in bx_images.SPLIT_NAMES)
nonempty_train = any((SPLITS_ROOT / "train").glob("*/*")) if (SPLITS_ROOT / "train").exists() else False
BASELINE_DATA_READY = False

if not TORCH_AVAILABLE:
    print("PyTorch/torchvision are not available. Intended baseline: ImageFolder + MobileNet/EfficientNet transfer learning.")
elif not split_dirs_ready or not nonempty_train:
    print("Split folders are missing or empty. Run the split command before building datasets.")
else:
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, models, transforms

    transform = transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    try:
        try:
            train_dataset = datasets.ImageFolder(SPLITS_ROOT / "train", transform=transform, allow_empty=True)
            val_dataset = datasets.ImageFolder(SPLITS_ROOT / "val", transform=transform, allow_empty=True)
        except TypeError:
            train_dataset = datasets.ImageFolder(SPLITS_ROOT / "train", transform=transform)
            val_dataset = datasets.ImageFolder(SPLITS_ROOT / "val", transform=transform)
    except Exception as exc:  # noqa: BLE001 - empty scaffold folders should not stop the notebook
        print(f"ImageFolder dataset build skipped: {type(exc).__name__}: {exc}")
        train_dataset = None
        val_dataset = None

    if train_dataset is not None and val_dataset is not None:
        BASELINE_DATA_READY = True
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

        weights = models.MobileNet_V3_Small_Weights.DEFAULT if ALLOW_PRETRAINED_DOWNLOAD else None
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, len(train_dataset.classes))

        print(f"ImageFolder classes: {len(train_dataset.classes)}")
        print(f"Train images: {len(train_dataset)}; validation images: {len(val_dataset)}")
        print("Model scaffold: MobileNetV3-small classifier head replaced for current classes.")

        if RUN_TRAINING_SMOKE_TEST:
            batch, target = next(iter(train_loader))
            output = model(batch)
            loss = nn.CrossEntropyLoss()(output, target)
            loss.backward()
            print(f"Smoke-test batch passed; loss={loss.item():.4f}")
        else:
            print("Training smoke test skipped. Set BIRDIDEX_RUN_TRAINING_SMOKE_TEST to run one batch.")


## 10. Evaluation Scaffold

This section defines executable evaluation helpers but does not fabricate results. Once a trained model exists, pass real labels, probabilities, and class names into these functions, then save the resulting metrics and plots under `data/reports/` or an explicit experiment folder.


In [ ]:
def top_k_accuracy(y_true: list[int], y_prob: list[list[float]], k: int) -> float | None:
    if not y_true or not y_prob:
        return None
    hits = 0
    for truth, probs in zip(y_true, y_prob, strict=False):
        top = sorted(range(len(probs)), key=lambda idx: probs[idx], reverse=True)[:k]
        hits += int(truth in top)
    return hits / len(y_true)


def weighted_f1(y_true: list[int], y_pred: list[int]) -> float | None:
    if not y_true or not y_pred:
        return None
    try:
        from sklearn.metrics import f1_score

        return float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    except Exception as exc:  # noqa: BLE001
        print(f"weighted F1 unavailable without scikit-learn: {type(exc).__name__}: {exc}")
        return None


def confusion_matrix_counts(y_true: list[int], y_pred: list[int], class_count: int) -> list[list[int]]:
    matrix = [[0 for _ in range(class_count)] for _ in range(class_count)]
    for truth, pred in zip(y_true, y_pred, strict=False):
        if 0 <= truth < class_count and 0 <= pred < class_count:
            matrix[truth][pred] += 1
    return matrix


def per_class_recall(y_true: list[int], y_pred: list[int], class_names: list[str]) -> list[dict[str, Any]]:
    rows = []
    for class_id, name in enumerate(class_names):
        positives = sum(1 for value in y_true if value == class_id)
        true_pos = sum(1 for truth, pred in zip(y_true, y_pred, strict=False) if truth == class_id and pred == class_id)
        rows.append({"class_id": class_id, "label": name, "recall": None if positives == 0 else true_pos / positives})
    return rows


def confidence_histogram(confidences: list[float], bins: int = 10) -> list[dict[str, Any]]:
    if not confidences:
        return []
    bucket_counts = Counter(min(bins - 1, max(0, int(score * bins))) for score in confidences)
    return [
        {"bin_start": index / bins, "bin_end": (index + 1) / bins, "count": bucket_counts[index]}
        for index in range(bins)
    ]


print("Evaluation helpers are ready. TODO: connect them to a trained model's saved predictions.")


## 11. Confidence Gate

The confidence gate turns top-k visual scores plus image-quality information into field UI states: high confidence, medium confidence, low confidence, multi-subject, or out-of-set. The system should abstain rather than force a wrong species ID when evidence is weak.


In [ ]:
example_top_k = [
    {"class_id": 71, "label": "galah", "confidence": 0.91},
    {"class_id": 97, "label": "little_corella", "confidence": 0.54},
    {"class_id": 128, "label": "rainbow_lorikeet", "confidence": 0.08},
]

decisions = [
    asdict(confidence_gate(example_top_k, image_quality_score=20.0)),
    asdict(confidence_gate(example_top_k, image_quality_score=1.0)),
    asdict(confidence_gate(example_top_k, multi_subject=True)),
    asdict(confidence_gate([{"class_id": None, "label": "out_of_set", "confidence": 0.72}])),
]
show_table(decisions)


## 12. Species Profile Integration

Profiles are loaded only from local structured data. Unknown natural-history fields remain `null` or explicit TODOs until curated from legitimate sources. This prevents the notebook from inventing habitat, behaviour, ID marks, similar species, or fun facts.


In [ ]:
profiles_path = EXPECTED_FILES["species_profiles.json"]
profiles = bx_profiles.load_species_profiles(profiles_path)

if not profiles:
    print("species_profiles.json is missing or empty. Build it with: uv run birdidex profiles build")
else:
    first_profile = profiles[0]
    by_id = bx_profiles.lookup_profile(profiles, class_id=first_profile.get("class_id"))
    by_label = bx_profiles.lookup_profile(profiles, label=str(first_profile.get("label")))
    print(f"Loaded {len(profiles)} species profiles from {profiles_path.relative_to(REPO_ROOT)}")
    expected_ui_fields = [
        "common_name",
        "scientific_name",
        "id_marks",
        "behaviour",
        "habitat",
        "similar_species",
        "fun_fact",
        "known_regions",
    ]
    rows = []
    for field in expected_ui_fields:
        rows.append({"field": field, "value": (by_id or {}).get(field)})
    show_table(rows)
    print(f"Lookup by class_id works: {by_id is not None}; lookup by label works: {by_label is not None}")


## 13. Observation Logging Schema

The observation schema is the future contract between capture/import, inference, UI review, and the offline observation log. The example below uses dummy non-private coordinates and paths.


In [ ]:
example_observation = ObservationRecord(
    observation_id="demo-observation-001",
    captured_at_utc=datetime.now(UTC).replace(microsecond=0).isoformat(),
    local_time="2026-07-08T12:00:00+10:00",
    timezone="Australia/Brisbane",
    season="winter",
    latitude=-27.4705,
    longitude=153.0260,
    gps_accuracy_m=25.0,
    region_guess="SEQ demo location",
    weather_summary="clear demo weather",
    image_path="data/images/raw/demo/example.jpg",
    crop_path="data/images/processed/demo/example_crop.jpg",
    predicted_class_id=71,
    predicted_label="galah",
    confidence=0.91,
    top_k_predictions=example_top_k,
    user_confirmed_class_id=None,
    user_feedback="pending_review",
)

show_table([example_observation.to_dict()])
schema = observation_json_schema()
print(f"Observation schema fields: {len(schema['properties'])}")


## 14. Device Runtime Logic

Runtime flow:

```text
Camera/import -> capture queue -> quality sorter -> detector/cropper -> classifier -> context prior -> confidence gate -> species profile -> UI stack -> observation log
```

Runtime state machine:

```text
IDLE -> PREVIEW -> CAPTURE/IMPORT -> QUALITY_SORT -> DETECT -> CLASSIFY -> GATE -> RESULT_UI -> SAVE/REJECT
```

Pi camera instant mode should prioritize a fast preview, a quick quality gate, and immediate result/abstain feedback. A7RV import mode can spend more time on batch quality checks, duplicate detection, manual crop review, and deferred confirmation because the source images are higher resolution and not necessarily captured inside the app.


## 15. CLI Integration

These are the ordered local commands that connect dataset collection, profile building, audit, tests, and notebook validation. The cell prints commands by default. It only executes them when `RUN_SAFE_CLI_COMMANDS` is set to `True` inside the notebook.


In [ ]:
CLI_COMMANDS = [
    {"command": "uv sync --all-groups", "why": "Install the repo, notebook, vision, training, inference, and test dependencies."},
    {"command": "uv run birdidex doctor", "why": "Check local package and class-index availability."},
    {"command": "uv run birdidex providers doctor", "why": "Report provider auth configuration without printing secrets or making live calls."},
    {"command": "uv run birdidex images scaffold", "why": "Create deterministic ImageFolder directories from class_index.json."},
    {"command": "uv run pytest", "why": "Run the offline unit test suite."},
    {"command": "uv run birdidex providers dry-run --provider ebird --species \"Rainbow Lorikeet\"", "why": "Validate eBird prior lookup for one species when an eBird key is configured."},
    {"command": "uv run birdidex providers dry-run --provider inaturalist --species \"Rainbow Lorikeet\" --limit 5", "why": "Preview iNaturalist open-license image metadata for one species."},
    {"command": "uv run birdidex images pipeline --species-limit 5 --per-class 25 --target-accepted 10 --max-edge 1024 --format jpg --quality 85", "why": "Run a small deterministic dataset collection/profile workflow."},
    {"command": "uv run birdidex images report", "why": "Regenerate image metadata reports."},
    {"command": "uv run birdidex audit dataset", "why": "Write dataset coverage and profile-gap audit artifacts."},
    {"command": "uv run birdidex profiles build", "why": "Build local species profiles from structured data without hallucinated facts."},
    {"command": "uv run python scripts/check_notebook_static.py notebooks/training/SEQ_BirdDex_Machine_Learning.ipynb", "why": "Check notebook JSON, required headings, local paths, and hard-coded secret patterns."},
]
show_table(CLI_COMMANDS)

RUN_SAFE_CLI_COMMANDS = False
if RUN_SAFE_CLI_COMMANDS:
    for item in CLI_COMMANDS:
        print(f"\n$ {item['command']}")
        result = subprocess.run(shlex.split(item["command"]), cwd=REPO_ROOT, text=True, capture_output=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"Command failed with exit code {result.returncode}: {item['command']}")
else:
    print("Command execution skipped. Set RUN_SAFE_CLI_COMMANDS = True to run this cell deliberately.")


## 16. Final Checklist

This checklist is intentionally factual. It reports what exists locally and leaves TODOs where real model training, evaluation, or curated profile facts are still pending.


In [ ]:
final_checks = [
    {"check": "Dataset class index exists", "status": EXPECTED_FILES["class_index.json"].exists()},
    {"check": "Metadata exists", "status": EXPECTED_FILES["image_records.jsonl"].exists()},
    {"check": "Profiles exist", "status": EXPECTED_FILES["species_profiles.json"].exists()},
    {"check": "Splits exist", "status": all((SPLITS_ROOT / split).exists() for split in bx_images.SPLIT_NAMES)},
    {"check": "Baseline can load data", "status": BASELINE_DATA_READY},
    {"check": "Evaluation skeleton ready", "status": callable(top_k_accuracy) and callable(weighted_f1)},
    {"check": "Device logic documented", "status": True},
    {"check": "No secrets intentionally stored in notebook", "status": True},
    {"check": "No private data paths intentionally hard-coded", "status": True},
]
show_table(final_checks)

intentional_todos = [
    {"cell": "Baseline Model Scaffold", "todo": "Run real training only after reviewed splits and experiment tracking are ready."},
    {"cell": "Evaluation Scaffold", "todo": "Fill metrics from saved predictions once a trained model exists."},
    {"cell": "Species Profile Integration", "todo": "Curate ID marks, habitat, behaviour, similar species, and fun facts from legitimate sources."},
    {"cell": "Device Runtime Logic", "todo": "Connect the state machine to the future camera/import UI and local observation writer."},
]
show_table(intentional_todos)
